### Data Cleaning and Text Preprocessing

**Input:** `../data/raw/ted_talks_selected.csv` (4,005 talks × 12 columns) <br>
**Output:** `../data/processed/ted_talks_cleaned.csv` <br>
**Based on:** findings from `02_eda_data_quality.ipynb`
- `speaker_occupation` - 522 missing (13.03%)
- 10 transcripts < 50 words (music/performance)
- Tags are comma-separated strings, free-form
- Views are heavily skewed

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [19]:
df = pd.read_csv('../data/raw/ted_talks_selected.csv')
print(df.columns.tolist())
df.head(3)

['talk_id', 'title', 'speaker', 'speaker_occupation', 'description', 'tags', 'transcript', 'url', 'views', 'duration', 'published_date', 'event']


,talk_id,title,speaker,speaker_occupation,description,tags,transcript,url,views,duration,published_date,event
0,1,Averting the climate crisis,Al Gore,climate advocate,With the same humor and humanity he exuded in ...,"alternative energy, cars, climate change, cult...","Thank you so much, Chris. And it's truly a gre...",https://www.ted.com/talks/al_gore_averting_the...,3523392,977,2006-06-27,TED2006
1,92,The best stats you've ever seen,Hans Rosling,global health expert; data visionary,You've never seen data presented like this. Wi...,"Africa, Asia, Google, demo, economics, global ...","About 10 years ago, I took on the task to teac...",https://www.ted.com/talks/hans_rosling_the_bes...,14501685,1190,2006-06-27,TED2006
2,7,Simplicity sells,David Pogue,technology columnist,New York Times columnist David Pogue takes aim...,"computers, entertainment, interface design, me...","(Music: ""The Sound of Silence,"" Simon & Garfun...",https://www.ted.com/talks/david_pogue_simplici...,1920832,1286,2006-06-27,TED2006


##### Dataset Review

In [20]:
print("Rows:", df.shape[0], "| Cols:", df.shape[1])
print("\nMissing (NaN) per column:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate talk_id:", df["talk_id"].duplicated().sum())
print("Duplicate url:", df["url"].duplicated().sum())

Rows: 4005 | Cols: 12

Missing (NaN) per column:
talk_id                 0
title                   0
speaker                 0
speaker_occupation    522
description             0
tags                    0
transcript              0
url                     0
views                   0
duration                0
published_date          0
event                   0
dtype: int64

Duplicate rows: 0
Duplicate talk_id: 0
Duplicate url: 0


##### Handle Missing Values

In [21]:
# text columns -> coerce to string, strip; empty strings / "nan" / "none" -> ""
text_cols = ["title", "description", "tags", "transcript", "url", "speaker", "speaker_occupation"]
for c in text_cols:
    df[c] = df[c].fillna("").astype(str).str.strip()
    df.loc[df[c].str.lower().isin(["", "nan", "none", "null"]), c] = ""

# empties before text normalization
print((df[text_cols] == "").sum())

title                   0
description             0
tags                    0
transcript              0
url                     0
speaker                 0
speaker_occupation    522
dtype: int64


In [22]:
# speaker_occupation: replace empties with "unknown"
df["speaker_occupation"] = df["speaker_occupation"].replace("", "unknown")

# remaining empties after text normalization
print((df[text_cols] == "").sum())

title                 0
description           0
tags                  0
transcript            0
url                   0
speaker               0
speaker_occupation    0
dtype: int64


##### Remove Duplicate Records

In [24]:
before = len(df)
dup_rows = df.duplicated().sum()
dup_ids  = df["talk_id"].duplicated().sum()
dup_urls = df["url"].duplicated().sum()
print(f"Duplicate rows: {dup_rows}")
print(f"Duplicate talk_id: {dup_ids}")
print(f"Duplicate url: {dup_urls}")

Duplicate rows: 0
Duplicate talk_id: 0
Duplicate url: 0


In [25]:
# There are no duplicate rows, but for consistency let's drop duplicates based on talk_id
# remove by talk_id (keep first)
df = df.drop_duplicates(subset="talk_id", keep="first").reset_index(drop=True)
after = len(df)

In [26]:
print(f"Rows before: {before}  |  Rows after: {after}  |  Removed: {before - after}")

Rows before: 4005  |  Rows after: 4005  |  Removed: 0


##### Clean Text Fields
Steps applied to `title`, `description`, `transcript`, `tags`:
1. Lowercase
2. Remove URLs (`http…`, `www…`)
3. Remove HTML tags (`<em>`, `<br>`) - some descriptions might contain them
4. Remove TED transcript noise markers: `(laughter)`, `(applause)`, `(music)`, `(audience)`, etc.
5. Replace non-alphanumeric characters with a space (keeps words, drops punctuation)
6. Collapse repeated whitespace and strip

In [27]:
import re

NOISE_RE = re.compile(r"\((?:laughter|applause|music|audience|mechanical noises)[^)]*\)", re.I)
HTML_RE  = re.compile(r"<[^>]+>")
URL_RE   = re.compile(r"https?://\S+|www\.\S+")
NON_ALPHA_RE = re.compile(r"[^a-z0-9\s]")
WS_RE    = re.compile(r"\s+")

In [36]:
def clean_text(t: str) -> str:
    if not isinstance(t, str):
        return ""
    t = t.lower()
    t = URL_RE.sub(" ", t)
    t = HTML_RE.sub(" ", t)
    t = NOISE_RE.sub(" ", t)
    t = NON_ALPHA_RE.sub(" ", t)
    t = WS_RE.sub(" ", t).strip()
    return t

for c in ["title", "description", "transcript"]:
    df[c + "_clean"] = df[c].apply(clean_text)

In [37]:
print(df[["title", "title_clean", "description", "description_clean", "transcript", "transcript_clean"]].head(3))

                             title                      title_clean  \
0      Averting the climate crisis      averting the climate crisis   
1  The best stats you've ever seen  the best stats you ve ever seen   
2                 Simplicity sells                 simplicity sells   

                                         description  \
0  With the same humor and humanity he exuded in ...   
1  You've never seen data presented like this. Wi...   
2  New York Times columnist David Pogue takes aim...   

                                   description_clean  \
0  with the same humor and humanity he exuded in ...   
1  you ve never seen data presented like this wit...   
2  new york times columnist david pogue takes aim...   

                                          transcript  \
0  Thank you so much, Chris. And it's truly a gre...   
1  About 10 years ago, I took on the task to teac...   
2  (Music: "The Sound of Silence," Simon & Garfun...   

                                    tran

In [40]:
# tags: split on comma, clean each, re-join -> already fine still done for consistency
def clean_tags(s: str) -> str:
    if not s:
        return ""
    parts = [clean_text(p) for p in s.split(",")]
    parts = [p for p in parts if p]
    return ", ".join(dict.fromkeys(parts))

df["tags_clean"] = df["tags"].apply(clean_tags)

In [41]:
print(df[["tags", "tags_clean"]].head(3))

                                                tags  \
0  alternative energy, cars, climate change, cult...   
1  Africa, Asia, Google, demo, economics, global ...   
2  computers, entertainment, interface design, me...   

                                          tags_clean  
0  alternative energy, cars, climate change, cult...  
1  africa, asia, google, demo, economics, global ...  
2  computers, entertainment, interface design, me...  


In [44]:
# Before/after examples for clean text fields

for col in ["title", "description", "transcript"]:
    print(f"\n===== {col} =====")
    print("ORIGINAL :", df[col].iloc[0][:200])
    print("CLEANED  :", df[col + "_clean"].iloc[0][:200])

print("\n===== tags =====")
print("ORIGINAL :", df["tags"].iloc[0])
print("CLEANED  :", df["tags_clean"].iloc[0])


===== title =====
ORIGINAL : Averting the climate crisis
CLEANED  : averting the climate crisis

===== description =====
ORIGINAL : With the same humor and humanity he exuded in "An Inconvenient Truth," Al Gore spells out 15 ways that individuals can address climate change immediately, from buying a hybrid to inventing a new, hott
CLEANED  : with the same humor and humanity he exuded in an inconvenient truth al gore spells out 15 ways that individuals can address climate change immediately from buying a hybrid to inventing a new hotter br

===== transcript =====
ORIGINAL : Thank you so much, Chris. And it's truly a great honor to have the opportunity to come to this stage twice; I'm extremely grateful. I have been blown away by this conference, and I want to thank all o
CLEANED  : thank you so much chris and it s truly a great honor to have the opportunity to come to this stage twice i m extremely grateful i have been blown away by this conference and i want to thank all of you

=====

##### Short-transcript flag

In [51]:
df["transcript_word_count"] = df["transcript_clean"].str.split().str.len()
df["is_short_transcript"] = df["transcript_word_count"] < 50
print("Short transcripts flagged:", df["is_short_transcript"].sum())

Short transcripts flagged: 10


##### Combined Text Field

In [52]:
# Plain space-joined baseline
df["combined_text"] = (
    df["title_clean"] + " " +
    df["tags_clean"].str.replace(",", " ", regex=False) + " " +
    df["description_clean"] + " " +
    df["transcript_clean"]
).str.replace(r"\s+", " ", regex=True).str.strip()

print(df["combined_text"].iloc[0][:300])

averting the climate crisis alternative energy cars climate change culture environment global issues science sustainability technology with the same humor and humanity he exuded in an inconvenient truth al gore spells out 15 ways that individuals can address climate change immediately from buying a 


In [53]:
# Prefixed version (preserves field boundaries for later weighting)
df["combined_text_prefixed"] = (
    "title: "       + df["title_clean"] + " " +
    "tags: "        + df["tags_clean"].str.replace(",", " ", regex=False) + " " +
    "description: " + df["description_clean"] + " " +
    "transcript: "  + df["transcript_clean"]
).str.replace(r"\s+", " ", regex=True).str.strip()

print(df["combined_text_prefixed"].iloc[0][:300])

title: averting the climate crisis tags: alternative energy cars climate change culture environment global issues science sustainability technology description: with the same humor and humanity he exuded in an inconvenient truth al gore spells out 15 ways that individuals can address climate change 


##### Save Cleaned Dataset

In [55]:
# full audit file (everything) 
df.to_csv("../data/processed/ted_talks_cleaned_all.csv", index=False)
print("Saved audit file:", "../data/processed/ted_talks_cleaned_all.csv", "| shape:", df.shape)
print(df.columns.tolist())

Saved audit file: ../data/processed/ted_talks_cleaned_all.csv | shape: (4005, 20)
['talk_id', 'title', 'speaker', 'speaker_occupation', 'description', 'tags', 'transcript', 'url', 'views', 'duration', 'published_date', 'event', 'title_clean', 'description_clean', 'transcript_clean', 'tags_clean', 'combined_text', 'combined_text_prefixed', 'transcript_word_count', 'is_short_transcript']


In [56]:
# final deliverable file, originals replaced by cleaned, reordered
df_final = df[[
    "talk_id", "title_clean", "speaker", "speaker_occupation",
    "description_clean", "tags_clean", "transcript_clean",
    "url", "views", "duration", "published_date", "event",
    "transcript_word_count", "is_short_transcript",
    "combined_text", "combined_text_prefixed",
]].rename(columns={
    "title_clean":       "title",
    "description_clean": "description",
    "tags_clean":        "tags",
    "transcript_clean":  "transcript",
})

final_path = "../data/processed/ted_talks_cleaned.csv"
df_final.to_csv(final_path, index=False)
print("Saved final file:", final_path, "| shape:", df_final.shape)
print(df_final.columns.tolist())

Saved final file: ../data/processed/ted_talks_cleaned.csv | shape: (4005, 16)
['talk_id', 'title', 'speaker', 'speaker_occupation', 'description', 'tags', 'transcript', 'url', 'views', 'duration', 'published_date', 'event', 'transcript_word_count', 'is_short_transcript', 'combined_text', 'combined_text_prefixed']


## Data Cleaning & Text Preprocessing Report

### Input / Output
- **Input:** `data/raw/ted_talks_selected.csv` (4,005 × 12) - untouched
- **Outputs:**
  - `data/processed/ted_talks_cleaned_all.csv` - full audit file (original + cleaned + flags + combined fields)
  - `data/processed/ted_talks_cleaned.csv` - final deliverable (`*_clean` renamed back to originals + `combined_text`, `combined_text_prefixed`, `is_short_transcript`)

### Before / After
| Metric | Before | After |
|---|---|---|
| Rows | 4,005 | 4,005 |
| Duplicate `talk_id` | 0 | 0 |
| Missing `speaker_occupation` | 522 (13.03%) | 0 (filled with `"unknown"`) |
| Empty `title` / `tags` / `transcript` | 0 | 0 |

No rows were dropped — every talk still carries usable text.

### Missing-Value Decisions
| Column | Decision | Reason |
|---|---|---|
| `speaker_occupation` | Fill with `"unknown"` | Optional context; dropping would remove 13% of talks |
| `title`, `description`, `tags`, `transcript`, `url` | Coerce to string, strip, empty/`"nan"` -> `""` | Normalize consistently; none were actually empty |
| Short transcripts (< 50 words) | Kept + flagged via `is_short_transcript` | Music/performance talks still have title/tags/description |

### Duplicates
`df.duplicated()` -> 0 rows · `talk_id` duplicates -> 0 · `url` duplicates -> 0. Dedupe step kept in the pipeline as a safe-guard for reruns.

### Text Cleaning Steps
Applied to `title`, `description`, `transcript`, `tags`:
1. Lowercase
2. Remove URLs and HTML tags
3. Remove TED transcript noise markers (`(laughter)`, `(applause)`, `(music)`, `(audience)`, `(mechanical noises)`)
4. Replace punctuation with spaces
5. Collapse whitespace and strip

Tags additionally deduped and re-joined with `, `.

### Before / After Examples
| Field | Original | Cleaned |
|---|---|---|
| `title` | `Averting the climate crisis` | `averting the climate crisis` |
| `tags` | `alternative energy, cars, climate change, …` | `alternative energy, cars, climate change, …` (lowercased/deduped) |
| `transcript` | `Thank you so much, Chris. … (Laughter) …` | `thank you so much chris …` (noise removed) |

Original text columns are preserved in `ted_talks_cleaned_all.csv` for audit.

### Combined Text Fields
Two variants built for downstream experiments:
- **`combined_text`** - plain space join: `title + tags + description + transcript` (baseline)
- **`combined_text_prefixed`** - field-prefixed: `title: … tags: … description: … transcript: …` (preserves field boundaries, allows weighting)

### Limitations
- Stopword removal and stemming/lemmatization were **not** done here - they belong in the `TfidfVectorizer` step so the CSV stays reusable.
- 10 talks have very short transcripts; flagged but not removed - accuracy of those recommendations will rely mostly on title/tags.
- TED noise regex targets known markers; some incidental parenthetical text may remain.
- English-only cleaning rules.